# Data generation

Runs the full LLM generation pipeline for each trait and exports `data/v2/prompts.json`.

Requires `OPENROUTER_API_KEY` in `.env` at the project root.

Set `resume: True` (default) to pick up from where a previous run left off without re-spending tokens.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))  # src/notebooks/ -> src/

from lib.data_generation_v1 import Pipeline

In [ ]:
CONFIG = {
    "output_dir": "data/v2",
    "traits": ["politeness", "hedging_confidence"],
    "models": {
        # Generator: creative, follows complex JSON schemas. Routed through google-vertex.
        "generator": {
            "model": "openrouter/google/gemini-2.0-flash-001",
            "family": "openai",
            "temperature": 0.5,
            "max_output_tokens": 2048,
            "litellm_kwargs": {
                "extra_body": {
                    "provider": {
                        "order": ["google-vertex"],
                        "allow_fallbacks": True,
                    }
                }
            },
        },
        # Judge: cross-family relative to the generator to limit leakage.
        "judge": {
            "model": "openrouter/deepseek/deepseek-v4-flash",
            "family": "deepseek",
            "temperature": 0.5,
            "max_output_tokens": 8192,
            "litellm_kwargs": {
                "extra_body": {
                    "provider": {
                        "order": ["alibaba"],
                        "allow_fallbacks": True,
                    }
                }
            },
        },
        # Tie-breaker judge: re-scores items the main judge places near the acceptance threshold.
        "tie_breaker_judge": {
            "model": "openrouter/openai/gpt-5-nano",
            "family": "openai",
            "temperature": 0.0,
            "max_output_tokens": 8192,
        },
    },
    "pipeline": {
        "scenarios_per_trait": 300,
        "scenario_batch_size": 5,
        "paraphrases_per_level": 3,
        "min_acceptance_score": 0.70,
        "max_duplicate_jaccard_scenarios": 0.85,
        "max_duplicate_jaccard": 0.85,
        "lexical_baseline_warning_accuracy": 0.55,
        "human_validation_fraction": 0.15,
        # 3 workers (not 4) avoids OpenRouter "peer closed connection" drops.
        "max_workers": 3,
        "resume": True,
        # Research-grade gates:
        "enable_intensity_scorer": True,
        "intensity_min_gap": 0.10,
        "max_length_ratio": 1.15,
        "max_retries": 2,
    },
    "validation": {
        "warn_if_same_family_generator_and_judge": True,
        "reject_if_same_family_generator_and_judge": False,
    },
}

In [ ]:
pipeline = Pipeline(CONFIG)

In [ ]:
for trait in CONFIG["traits"]:
    print(f"\n{'='*60}\n  {trait}\n{'='*60}")
    pipeline.make_all(trait)

In [ ]:
pipeline.export_prompts()
print(f"\nDone. {CONFIG[\"output_dir\"]}/prompts.json is ready.")